**SEANCE :** 1
**NOM :** YEDONOU Alvine
**PORTEUR # :** P1
**ETAPES ASSIGNEES :** 1.1 Inventorier les tables → 1.8 Fuites  *(version corrigée)*

# P1 — Inventaire des tables (1.1) et fuites (1.8)
Phase 1 du projet *Santander Satisfaction* : connaître les données avant de modéliser.
Chaque tâche suit le même plan : **la question**, **la mesure** (le code), **le constat**, **la décision**.

Ce carnet est **autonome** : il trouve `train.csv` et `a_decider.csv` tout seul (variable `DONNEES_PROJET`, dossier `data/` ou `../data`, ou `/kaggle/input`). Exécuter les cellules **dans l'ordre**, de haut en bas.

**Ce qui change par rapport à la première version**
- `train_complet` est maintenant défini **avant** le 1.1 : le carnet se relance de haut en bas.
- Les doublons sont comptés **sans la colonne `ID`** : avec elle, le résultat est toujours 0 puisque l'`ID` est unique.
- Le test de « pureté » du 1.8 est **retiré** : il signalait 38 colonnes alors qu'aucune n'est une fuite (voir plus bas).
- Le 1.8 contrôle désormais les identifiants, la position dans le fichier, les lignes communes, la force de chaque variable, la ressemblance entre `train` et `a_decider`, et se termine par un **verdict**.

## 0. Réglages et chargement

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

CIBLE, ID = "TARGET", "ID"
GRAINE = 42
PLANCHER = 0.6890          # AUC de la règle métier (README)


def trouver_data():
    """Cherche le dossier qui contient train.csv ET a_decider.csv."""
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici / "data", ici.parent / "data", ici.parent.parent / "data", ici]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        pistes += [p.parent for p in kaggle.rglob("train.csv")]
    for p in pistes:
        if (p / "train.csv").is_file() and (p / "a_decider.csv").is_file():
            return p
    raise FileNotFoundError("train.csv / a_decider.csv introuvables. Essayé : " + ", ".join(map(str, pistes[:6])))


DATA = trouver_data()
FICHIERS = {"train.csv": "entraînement (avec la cible)", "a_decider.csv": "à prédire (sans la cible)"}

train_complet = pd.read_csv(DATA / "train.csv")
test_complet = pd.read_csv(DATA / "a_decider.csv")      # = a_decider.csv
VARIABLES = [c for c in train_complet.columns if c not in (ID, CIBLE)]
y = train_complet[CIBLE]

print("dossier de données :", DATA)
print("train :", train_complet.shape, "| a_decider :", test_complet.shape)

dossier de données : /home/vino-yd/santander-satisfaction/santander-satisfaction/data
train : (60816, 371) | a_decider : (15204, 370)


## 1.1 — Inventorier les tables
**Question** : quelles tables avons-nous, de quelle taille, avec quel rôle, et quelle est leur clé ?
**Mesure** : dimensions, rôle, clé, manquants, mémoire ; puis comparaison des colonnes entre les deux tables.

In [2]:
tables = {"train.csv": train_complet, "a_decider.csv": test_complet}

inventaire = pd.DataFrame(
    {
        nom: {
            "rôle": FICHIERS[nom],
            "taille du fichier (Mo)": round((DATA / nom).stat().st_size / 1e6, 1),
            "lignes": len(df),
            "colonnes": df.shape[1],
            "clé": ID,
            "ID unique": bool(df[ID].is_unique),
            "ID manquants": int(df[ID].isna().sum()),
            "cible présente": CIBLE in df.columns,
            "valeurs manquantes (NaN)": int(df.isna().sum().sum()),
            "mémoire (Mo)": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
        }
        for nom, df in tables.items()
    }
)
display(inventaire)

,train.csv,a_decider.csv
rôle,entraînement (avec la cible),à prédire (sans la cible)
taille du fichier (Mo),60.5000,15.1000
lignes,60816,15204
colonnes,371,370
clé,ID,ID
ID unique,True,True
ID manquants,0,0
cible présente,True,False
valeurs manquantes (NaN),0,0
mémoire (Mo),172.1000,42.9000


In [3]:
cols_tr, cols_de = list(train_complet.columns), list(test_complet.columns)
communes = [c for c in cols_tr if c in cols_de]

print("Colonnes dans train seulement      :", sorted(set(cols_tr) - set(cols_de)))
print("Colonnes dans a_decider seulement  :", sorted(set(cols_de) - set(cols_tr)))
print("Colonnes communes                  :", len(communes))
print("Même ordre des colonnes communes   :", communes == cols_de)

types_diff = [c for c in communes if train_complet[c].dtype != test_complet[c].dtype]
print("Colonnes de type différent entre les tables :", len(types_diff), types_diff[:5])
print("Types dans train :", train_complet.dtypes.astype(str).value_counts().to_dict())

print(f"\nCible : valeurs {sorted(int(v) for v in y.unique())}, "
      f"{int(y.sum())} mécontents sur {len(y)} = {100 * y.mean():.2f} %")
print(f"Mécontents attendus dans a_decider au même taux : {round(y.mean() * len(test_complet))}")

# Doublons : SANS la colonne ID (avec elle, le résultat serait toujours 0)
dup_avec_id = int(train_complet.duplicated().sum())
dup_train = int(train_complet.duplicated(subset=VARIABLES).sum())
dup_test = int(test_complet.duplicated(subset=VARIABLES).sum())
print(f"\nLignes dupliquées dans train, ID compris           : {dup_avec_id}  (toujours 0 : l'ID est unique)")
print(f"Lignes dupliquées dans train, variables seulement   : {dup_train}")
print(f"Lignes dupliquées dans a_decider, variables seules  : {dup_test}")

Colonnes dans train seulement      : ['TARGET']
Colonnes dans a_decider seulement  : []
Colonnes communes                  : 370
Même ordre des colonnes communes   : True
Colonnes de type différent entre les tables : 0 []
Types dans train : {'int64': 260, 'float64': 111}

Cible : valeurs [0, 1], 2406 mécontents sur 60816 = 3.96 %
Mécontents attendus dans a_decider au même taux : 602

Lignes dupliquées dans train, ID compris           : 0  (toujours 0 : l'ID est unique)
Lignes dupliquées dans train, variables seulement   : 3788
Lignes dupliquées dans a_decider, variables seules  : 684


**Constat**
- `train.csv` : 60 816 clients × 371 colonnes (`ID`, `TARGET` et 369 variables). `a_decider.csv` : 15 204 clients × 370 colonnes. La clé `ID` est unique et jamais vide dans les deux tables.
- Les deux tables ont **370 colonnes communes** ; la seule différence est `TARGET`, présente uniquement dans `train`. Aucune valeur manquante (`NaN`).
- Types dans `train` : 260 colonnes entières et 111 décimales. Les deux tables tiennent en mémoire (environ 172 Mo et 43 Mo), on les lit donc en entier.
- `TARGET` vaut 3,96 % de 1 (2 406 mécontents), soit environ 602 mécontents attendus dans `a_decider`.
- **Doublons** : 0 si l'on compte l'`ID`, mais **3 788** lignes de `train` ont des variables identiques à une autre ligne (mesure faite en séance 2). Le chiffre « 0 » de la première version était donc trompeur.

**Décision**
- Lire les tables en entier ; `ID` sert de clé et n'est **jamais** une variable.
- Transmettre les doublons à **P4 (1.7)** : il doit les compter sans l'`ID`.
- Les valeurs codées (`-999999`, `9999999999`) ne sont pas des manquants `NaN` : elles relèvent de **P3 (1.6)**.

## 1.8 — Fuites
**Question** : qu'est-ce qui permettrait à un modèle de « tricher », c'est-à-dire d'avoir un bon score en validation et d'être inutile sur `a_decider` ?
**Mesure** : six contrôles, puis un verdict.

> **Pourquoi l'ancien test de « pureté » est retiré.** Il signalait toute colonne ayant une valeur présente chez au moins 20 clients sans aucun mécontent. Avec 96 % de satisfaits, cela arrive sur des colonnes banales : 38 colonnes ont été signalées, aucune n'est une fuite. P3 le montre au bloc 1.4 : sur les 110 colonnes presque constantes, 94 n'ont aucun mécontent parmi leurs clients rares, ce qui est compatible avec le hasard.

In [4]:
from sklearn.metrics import roc_auc_score

def auc_sym(cible, score):
    """AUC ramenée entre 0,5 et 1 (on ignore le sens de la relation)."""
    a = roc_auc_score(cible, score)
    return max(a, 1 - a)

# --- (a) Identifiants communs aux deux tables
communs = set(train_complet[ID]) & set(test_complet[ID])

# --- (b) L'ID et la position dans le fichier prédisent-ils la cible ?
auc_id = auc_sym(y, train_complet[ID])
auc_position = auc_sym(y, np.arange(len(train_complet)))

# --- (c) Lignes de a_decider dont les variables existent déjà dans train
h_train = pd.util.hash_pandas_object(train_complet[VARIABLES], index=False)
h_test = pd.util.hash_pandas_object(test_complet[VARIABLES], index=False)
connus = h_test.isin(h_train.values)
n_connus = int(connus.sum())

# --- (d) Noms de colonnes suspects
mots_suspects = ["target", "label", "outcome", "result", "response", "fraud", "satisfaction"]
noms_suspects = [c for c in VARIABLES if any(m in c.lower() for m in mots_suspects)]

print(f"(a) ID communs aux deux tables                  : {len(communs)}")
print(f"(b) AUC de l'ID seul                            : {auc_id:.4f}")
print(f"    AUC de la position dans le fichier          : {auc_position:.4f}")
print(f"(c) Lignes de a_decider déjà dans train         : {n_connus} sur {len(test_complet)} "
      f"({100 * n_connus / len(test_complet):.2f} %)")
print(f"(d) Noms de colonnes suspects                   : {noms_suspects}")

(a) ID communs aux deux tables                  : 0
(b) AUC de l'ID seul                            : 0.5076
    AUC de la position dans le fichier          : 0.5045
(c) Lignes de a_decider déjà dans train         : 1130 sur 15204 (7.43 %)
(d) Noms de colonnes suspects                   : []


In [5]:
# --- (e) Force de chaque variable prise seule : une variable qui prédit presque tout est suspecte
resultats = []
for c in VARIABLES:
    x = train_complet[c]
    if x.nunique(dropna=False) < 2:
        continue                                   # colonne constante : rien à tester
    x = x.fillna(x.median())
    resultats.append((c, auc_sym(y, x)))

auc_vars = (pd.DataFrame(resultats, columns=["variable", "AUC seule"])
            .sort_values("AUC seule", ascending=False).set_index("variable"))
auc_max = float(auc_vars["AUC seule"].iloc[0])

print(f"{len(auc_vars)} variables testées. Repère : le plancher métier est {PLANCHER}.")
display(auc_vars.head(10))

333 variables testées. Repère : le plancher métier est 0.689.


,AUC seule
variable,
var15,0.7010
saldo_var30,0.6952
num_meses_var5_ult3,0.6930
saldo_var42,0.6801
num_var30,0.6795
saldo_var5,0.6785
saldo_medio_var5_hace2,0.6785
saldo_medio_var5_ult1,0.6764
saldo_medio_var5_ult3,0.6752


In [6]:
# --- (f) train et a_decider se ressemblent-ils ? (un modèle essaie de les distinguer)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X_adv = pd.concat([train_complet[VARIABLES], test_complet[VARIABLES]], ignore_index=True)
y_adv = np.r_[np.zeros(len(train_complet)), np.ones(len(test_complet))]
p_adv = cross_val_predict(
    HistGradientBoostingClassifier(max_iter=50, random_state=GRAINE),
    X_adv, y_adv,
    cv=StratifiedKFold(3, shuffle=True, random_state=GRAINE),
    method="predict_proba",
)[:, 1]
auc_adv = float(roc_auc_score(y_adv, p_adv))
print(f"(f) AUC pour distinguer train de a_decider : {auc_adv:.4f}  (0,5 = impossible à distinguer)")

(f) AUC pour distinguer train de a_decider : 0.4986  (0,5 = impossible à distinguer)


### Verdict, une ligne par contrôle

In [7]:
def verdict_auc(a):
    return "ALERTE" if a >= 0.90 else ("À EXAMINER" if a >= 0.75 else "OK")

controles = [
    ("a. ID communs train / a_decider", str(len(communs)), "0",
     "OK" if len(communs) == 0 else "ALERTE"),
    ("b. AUC de l'ID seul", f"{auc_id:.4f}", "proche de 0,5 (< 0,55)",
     "OK" if auc_id < 0.55 else "ALERTE"),
    ("b. AUC de la position dans le fichier", f"{auc_position:.4f}", "proche de 0,5 (< 0,55)",
     "OK" if auc_position < 0.55 else "ALERTE"),
    ("c. Lignes de a_decider déjà dans train", str(n_connus), "moins de 1 % de a_decider",
     "OK" if n_connus < 0.01 * len(test_complet) else "À EXAMINER"),
    ("d. Noms de colonnes suspects", str(len(noms_suspects)), "0",
     "OK" if not noms_suspects else "À EXAMINER"),
    ("e. Plus forte AUC d'une variable seule", f"{auc_max:.4f}", "< 0,75 (alerte dès 0,90)",
     verdict_auc(auc_max)),
    ("f. AUC train contre a_decider", f"{auc_adv:.4f}", "proche de 0,5 (< 0,60)",
     "OK" if auc_adv < 0.60 else "À EXAMINER"),
]
tableau = pd.DataFrame(controles, columns=["contrôle", "résultat", "attendu si sain", "verdict"])
display(tableau)

pas_ok = tableau[tableau["verdict"] != "OK"]
if pas_ok.empty:
    conclusion = ("Aucune fuite détectée par ces six contrôles. Ce n'est pas une preuve d'absence de fuite, "
                  "seulement l'absence de signal sur ce qu'on a mesuré.")
else:
    conclusion = "Points à examiner : " + " ; ".join(pas_ok["contrôle"]) + "."
display(Markdown("**Conclusion automatique :** " + conclusion))

,contrôle,résultat,attendu si sain,verdict
0,a. ID communs train / a_decider,0,0,OK
1,b. AUC de l'ID seul,0.5076,"proche de 0,5 (< 0,55)",OK
2,b. AUC de la position dans le fichier,0.5045,"proche de 0,5 (< 0,55)",OK
3,c. Lignes de a_decider déjà dans train,1130,moins de 1 % de a_decider,À EXAMINER
4,d. Noms de colonnes suspects,0,0,OK
5,e. Plus forte AUC d'une variable seule,0.7010,"< 0,75 (alerte dès 0,90)",OK
6,f. AUC train contre a_decider,0.4986,"proche de 0,5 (< 0,60)",OK


**Conclusion automatique :** Points à examiner : c. Lignes de a_decider déjà dans train.

In [8]:
# Texte prêt à recopier dans le rendu (les chiffres viennent des cellules ci-dessus)
meilleure = auc_vars.index[0]
display(Markdown(f"""**Constat 1.8**
- {len(communs)} `ID` commun aux deux tables ; AUC de l'`ID` {auc_id:.2f} et de la position dans le fichier {auc_position:.2f} (0,5 = aucun lien avec la cible).
- {n_connus} lignes de `a_decider` ({100 * n_connus / len(test_complet):.2f} %) ont des variables identiques à une ligne de `train`.
- Variable la plus liée à la cible : `{meilleure}` (AUC {auc_max:.3f}), à comparer au plancher métier {PLANCHER}.
- Un modèle distingue `train` de `a_decider` avec une AUC de {auc_adv:.2f} (0,5 = tables semblables).
- {conclusion}
"""))

**Constat 1.8**
- 0 `ID` commun aux deux tables ; AUC de l'`ID` 0.51 et de la position dans le fichier 0.50 (0,5 = aucun lien avec la cible).
- 1130 lignes de `a_decider` (7.43 %) ont des variables identiques à une ligne de `train`.
- Variable la plus liée à la cible : `var15` (AUC 0.701), à comparer au plancher métier 0.689.
- Un modèle distingue `train` de `a_decider` avec une AUC de 0.50 (0,5 = tables semblables).
- Points à examiner : c. Lignes de a_decider déjà dans train.


**Décision**
- `ID` et la position des lignes ne sont **jamais** des variables du modèle.
- Tout prétraitement (moyenne d'imputation, normalisation, choix de variables) est **calculé sur l'apprentissage seulement**, jamais sur `train` et `a_decider` réunis.
- Si un contrôle est « ALERTE » : retirer la colonne concernée et l'écrire dans le journal des décisions. S'il est « À EXAMINER » : le signaler au groupe avant de modéliser.
- Les doublons de `train` peuvent gonfler la validation s'ils tombent des deux côtés du découpage : à mesurer avec **P4 (1.7)** et à tenir pour le découpage 2.1.